In [ ]:
%pip install . --user

In [1]:
import pandas as pd
import numpy as np
from jupyter_woe_binner import BinningWidget, BinningWidgetList

In [2]:
np.random.seed(42)
n = 2000

amount = np.random.gamma(2, 50000, n)
credit_usage = np.random.uniform(0, 1, n)
income = np.random.lognormal(10, 1, n)
age = np.random.randint(20, 70, n)

prob = (0.3
        - 0.15 * (amount - amount.min()) / (amount.max() - amount.min())
        + 0.1 * credit_usage
        - 0.1 * (income - income.min()) / (income.max() - income.min()))
prob = np.clip(prob, 0.05, 0.95)
target = (np.random.rand(n) < prob).astype(int)

df = pd.DataFrame({
    'adj_finance_amt': amount,
    'credit_6m_usage': credit_usage,
    'annual_income': income,
    'age': age,
    'target': target
})

## 操作说明

| 操作 | 方式 |
|------|------|
| 选择箱子 | 在 Select 多选框中按住 Ctrl 点击，或点击图表柱子 |
| 合并连续箱 | 选中 2 个或以上连续箱 → 点击 **⬌ Merge** 或按 `Ctrl+⇧W` |
| IV 最优分裂 | 选中 1 个箱 → 点击 **⬍ Split** 或按 `Ctrl+⇧Q` |
| 💥 BOOM! 爆炸分裂 | 选中 1 个箱 → 点击 **💥 BOOM!** 或按 `Ctrl+⇧B`，按 1% 百分位强制分裂至最多分箱 |
| 确认分箱 | 点击 **✓ Confirm** |
| 重置 | 点击 **↺ Reset** |

## 1. 单变量分箱

In [ ]:
widget = BinningWidget(df, var_name='adj_finance_amt', 
                       target_name='target',
                       event_flag=1, 
                       non_event_flag=0, 
                       max_bins=6)
widget.display()

In [ ]:
widget.bins

## 2. 单变量分箱（自定义初始分箱）

In [ ]:
# initial_bins: 自定义初始分箱边界（不需要包含 -inf 和 inf，系统会自动添加）
widget_init = BinningWidget(df, var_name='adj_finance_amt',
                            target_name='target',
                            event_flag=1,
                            non_event_flag=0,
                            initial_bins=[-np.inf, 50000, 80000,100000, 150000, 200000,np.inf])
widget_init.display()

In [ ]:
widget_init.bins

## 3. 单变量分箱（含特殊值 + 最小箱占比限制）

In [ ]:
# 模拟特殊值：-99999 代表缺失，-99998 代表未知
df_spc = df.copy()
df_spc.loc[np.random.choice(n, 80, replace=False), 'adj_finance_amt'] = -99999
df_spc.loc[np.random.choice(n, 50, replace=False), 'adj_finance_amt'] = -99998

# min_split_pct=0.05: 分裂后每个箱至少占比 5%
widget_spc = BinningWidget(df_spc, var_name='adj_finance_amt',
                           target_name='target',
                           event_flag=1,
                           non_event_flag=0,
                           max_bins=6,
                           spc_values=[-99999, -99998],
                           min_split_pct=0.05)
widget_spc.display()

In [ ]:
widget_spc.bins

## 4. 多变量批量分箱

In [ ]:
widget_list = BinningWidgetList(df, 
    var_name=['adj_finance_amt', 'credit_6m_usage', 'annual_income', 'age'],
    target_name='target',
    event_flag=1,
    non_event_flag=0,
    max_bins=6)
widget_list.display()

In [ ]:
widget_list.bins

## 5. 多变量批量分箱（自定义初始分箱 + 特殊值 + 最小箱占比）

In [3]:
# initial_bin_dir: 为每个变量设置个性化初始分箱边界
# 不在 initial_bin_dir 中的变量将使用 max_bins 等频分箱
initial_bin_dir = {
    'adj_finance_amt': [-np.inf,50000, 100000, 150000, 200000,np.inf],
    'credit_6m_usage': [-np.inf,0.2, 0.4, 0.6, 0.8,np.inf],
    'annual_income':   [-np.inf,20000, 50000, 80000, 120000,np.inf],
}

df_spc2 = df.copy()
for col in ['adj_finance_amt', 'credit_6m_usage', 'annual_income', 'age']:
    df_spc2.loc[np.random.choice(n, 60, replace=False), col] = -99999
    df_spc2.loc[np.random.choice(n, 40, replace=False), col] = -99998

widget_list_init = BinningWidgetList(df_spc2,
    var_name=['adj_finance_amt', 'credit_6m_usage', 'annual_income', 'age'],
    target_name='target',
    event_flag=1,
    non_event_flag=0,
    max_bins=6,
    spc_values=[-99999, -99998],
    min_split_pct=0.02,
    initial_bin_dir=None)
widget_list_init.display()

In [ ]:
widget_list_init.bins